# MNIST digit recognition — CNN + elastic-distortion augmentation (99.62 % test accuracy)

A compact CNN trained on a heavily augmented MNIST training set, following the augmentation ideas of
Simard et al. (2003) "Best practices for convolutional neural networks applied to visual document analysis".

**Pre-processing**
* centre crop 28×28 → 20×20 (MNIST digits occupy the central 20×20 box) and per-image mean subtraction.

**Augmentation (60 k → 1.2 M training images)**
1. affine: one random rotation (±20°), one random shear (0.2) and one random shift (±20 %) per image → ×4;
2. **elastic distortions**: 4 random displacement fields per image, smoothed with a Gaussian (σ = 4) and scaled by
   α = 6, applied with bilinear interpolation → ×5.

**Model:** Conv(20, 3×3) → MaxPool → Conv(40, 3×3) → MaxPool → Dense(640) → Dropout(0.5) → Dense(1000) →
Dropout(0.5) → Softmax(10); SGD with Nesterov momentum 0.95, lr 0.01, 30 epochs, batch 128.

**Result (logged output below):** test loss 0.0214, **test accuracy 0.9962 — 38 errors on the 10 000 test images.**

> The experiment was originally run with Keras 2.1.5 on TensorFlow 1.15 (the training log below is from that run).
> The code has since been ported to `tf.keras` (TensorFlow 2.15 / Keras 2 API); the model and augmentation are unchanged.


In [ ]:
%pip install -q "tensorflow==2.15.*" scipy

In [ ]:
import random

import numpy as np
from scipy.ndimage import gaussian_filter, map_coordinates
from tensorflow.keras import optimizers
from tensorflow.keras.datasets import mnist
from tensorflow.keras.layers import Activation, Conv2D, Dense, Dropout, Flatten, MaxPooling2D
from tensorflow.keras.models import Sequential
from tensorflow.keras.preprocessing.image import random_rotation, random_shear, random_shift
from tensorflow.keras.utils import to_categorical

In [ ]:
def augment_data(dataset, dataset_labels, augmentation_factor=1, use_random_rotation=True,
                 use_random_shear=True, use_random_shift=True):
    """Keep every original image and add `augmentation_factor` rotated, sheared and shifted copies."""
    augmented_image = []
    augmented_image_labels = []
    for num in range(dataset.shape[0]):
        augmented_image.append(dataset[num])
        augmented_image_labels.append(dataset_labels[num])
        for _ in range(augmentation_factor):
            if use_random_rotation:
                augmented_image.append(random_rotation(dataset[num], 20, row_axis=0, col_axis=1, channel_axis=2))
                augmented_image_labels.append(dataset_labels[num])
            if use_random_shear:
                augmented_image.append(random_shear(dataset[num], 0.2, row_axis=0, col_axis=1, channel_axis=2))
                augmented_image_labels.append(dataset_labels[num])
            if use_random_shift:
                augmented_image.append(random_shift(dataset[num], 0.2, 0.2, row_axis=0, col_axis=1, channel_axis=2))
                augmented_image_labels.append(dataset_labels[num])
    return np.array(augmented_image), np.array(augmented_image_labels)


def crop(img, size, offset=4):
    """Fixed centre crop (offset 4 turns a 28x28 MNIST digit into its central 20x20 box)."""
    return img[offset:offset + size, offset:offset + size]


def elastic_transform(image, alpha, sigma, random_state=None):
    """Elastic distortion (Simard et al., 2003): smoothed random displacement field scaled by alpha."""
    if random_state is None:
        random_state = np.random.RandomState(None)
    shape = image.shape
    dx = gaussian_filter((random_state.rand(*shape) * 2 - 1), sigma, mode="constant", cval=0) * alpha
    dy = gaussian_filter((random_state.rand(*shape) * 2 - 1), sigma, mode="constant", cval=0) * alpha
    x, y = np.meshgrid(np.arange(shape[0]), np.arange(shape[1]))
    indices = np.reshape(y + dy, (-1, 1)), np.reshape(x + dx, (-1, 1))
    return map_coordinates(image, indices, order=1).reshape(shape)


def get_elastics(images, labels, alpha, sigma, n_distortions=4):
    """Return every image followed by `n_distortions` elastically distorted copies."""
    augmented = []
    augmented_labels = []
    for i in range(len(images)):
        news = np.array([elastic_transform(images[i][0], alpha, sigma) for _ in range(n_distortions)])
        news = np.append(news, images[i][0])
        augmented.append(news)
        augmented_labels.append([labels[i]] * (n_distortions + 1))
    n = (n_distortions + 1) * len(images)
    return (np.reshape(augmented, (n, images.shape[2], images.shape[3])),
            np.reshape(augmented_labels, n))


def augment_elastic(images, labels, alpha, sigma):
    images = images.reshape(images.shape[0], 1, 20, 20)
    deformed, new_labels = get_elastics(images, labels, alpha, sigma)
    return deformed.reshape(deformed.shape[0], 20, 20, 1), new_labels

In [ ]:
(X_train, y_train), (X_test, y_test) = mnist.load_data()

X_train = X_train.reshape(X_train.shape[0], 28, 28, 1).astype('float32') / 255
X_test = X_test.reshape(X_test.shape[0], 28, 28, 1).astype('float32') / 255

# central 20x20 crop + per-image mean subtraction
X_train = np.array([crop(img, 20) for img in X_train])
X_test = np.array([crop(img, 20) for img in X_test])
X_train = np.array([img - np.mean(img) for img in X_train])
X_test = np.array([img - np.mean(img) for img in X_test])

# 60k -> 240k (affine) -> 1.2M (elastic)
X_train_random, y_train_random = augment_data(X_train, y_train, augmentation_factor=1)
X_train_2, y_train_2 = augment_elastic(X_train_random, y_train_random, alpha=6, sigma=4)
Y_train_2 = to_categorical(y_train_2, 10)
Y_test = to_categorical(y_test, 10)
print(X_train_2.shape, Y_train_2.shape)

In [ ]:
def build_model(input_shape):
    model = Sequential([
        Conv2D(20, (3, 3), padding='same', input_shape=input_shape),
        Activation('relu'),
        MaxPooling2D(pool_size=(2, 2)),
        Conv2D(40, (3, 3), padding='same'),
        Activation('relu'),
        MaxPooling2D(pool_size=(2, 2)),
        Flatten(),
        Dense(640), Activation('relu'), Dropout(0.5),
        Dense(1000), Activation('relu'), Dropout(0.5),
        Dense(10), Activation('softmax'),
    ])
    sgd = optimizers.legacy.SGD(learning_rate=0.01, decay=1e-6, momentum=0.95, nesterov=True)
    model.compile(loss='categorical_crossentropy', optimizer=sgd, metrics=['accuracy'])
    return model


def experiment(X_train, Y_train, file, epochs=30):
    model = build_model(X_train.shape[1:])
    model.fit(X_train, Y_train, batch_size=128, epochs=epochs, verbose=1)

    score = model.evaluate(X_test, Y_test, verbose=0)
    print(score)  # [test loss, test accuracy]

    preds = np.argmax(model.predict(X_test, verbose=0), axis=1)
    mistakes = np.where(preds != y_test)[0]
    print("Number of errors:")
    print(len(mistakes))
    print(mistakes)

    model.save(file)
    return model


model = experiment(X_train_2, Y_train_2, "mnist_cnn_elastic.keras")

... (epochs 1-29 omitted) ...
Epoch 30/30
1200000/1200000 [==============================] - 750s 625us/step - loss: 0.0024 - acc: 0.9992
[0.021371482182448335, 0.9962]
(10000,)
Number of errors:
38
[8.000e+00 6.590e+02 7.260e+02 9.470e+02 1.226e+03 1.242e+03 1.247e+03
 1.260e+03 2.035e+03 2.044e+03 2.130e+03 2.280e+03 2.293e+03 2.447e+03
 2.462e+03 2.488e+03 2.597e+03 2.654e+03 2.771e+03 2.896e+03 2.939e+03
 3.422e+03 4.369e+03 4.571e+03 4.620e+03 4.740e+03 4.761e+03 4.823e+03
 5.457e+03 5.955e+03 6.597e+03 6.625e+03 8.279e+03 8.326e+03 9.642e+03
 9.692e+03 9.729e+03 9.792e+03]
[38.]
